# DeFi Aggregate — Multi-Chain Activity & Outlook

Track **cross-chain DeFi** (TVL, DEX volume, fees) across major EVMs + Solana,
detect **regime / trend changes**, overlay **legislation and known shocks**, and
answer **can we track Trump's crypto assets?** ($TRUMP, WLFI, protocol TVL, public
token mints — **not** DJT equity).

**Thesis.** DeFi activity is a multi-chain claim on risk appetite and on-chain
liquidity. Aggregate TVL, DEX volume, and protocol fees move together through
expansions and contractions; chain share rotates with L2 / Solana cycles.
Regulatory and market shocks (UST, FTX, MiCA, FIT21, inauguration) imprint
step-changes that simple momentum ignores unless marked.

**Pipeline (run top-to-bottom):**
1. Setup + cache dir (`data/defi_cache`)
2. DefiLlama global + per-chain TVL / DEX volume / fees (keyless; synthetic fallback)
3. Market context from DuckDB (BTC/ETH) + optional FRED liquidity
4. Weekly features, activity composite, regime shading
5. Event overlays from `dbt/seeds/events.csv` + pre/post window stats
6. Outlook snapshot (rule-based)
7. Trump crypto panel ($TRUMP / WLFI / DefiLlama / wallet seeds)
8. Caveats

Graceful degradation: missing API keys or network → seeded synthetic series so the
notebook still completes. Tags print `REAL` / `SYNTHETIC` / `MISSING`.


In [ ]:
from __future__ import annotations

import json
import os
import time
import warnings
from datetime import UTC, date, datetime, timedelta
from pathlib import Path
from typing import Any

import httpx
import numpy as np
import plotly.graph_objects as go
import polars as pl
from dotenv import load_dotenv
from plotly.subplots import make_subplots

from ccquant.forecasting import load_daily_panel, load_signals_panel

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# --- Config ----------------------------------------------------------------
_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root.parent != _root:
    _root = _root.parent

load_dotenv(_root / ".env", override=True)

DB_PATH = os.environ.get("CCQUANT_DB", str(_root / "data" / "ccquant.duckdb"))
FRED_API_KEY = os.environ.get("FRED_API_KEY", "").strip()
CG_DEMO_API_KEY = os.environ.get("CG_DEMO_API_KEY", "").strip()
COINGECKO_ATTRIBUTION = (
    "Data provided by CoinGecko (https://www.coingecko.com)"
)

CACHE_DIR = _root / "data" / "defi_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_MAX_AGE_H = 12.0
EVENTS_CSV = _root / "dbt" / "seeds" / "events.csv"
WALLET_SEED = _root / "config" / "seeds" / "wallet_registry_seed.csv"

WEEKLY_FREQ = "1w"
MOM_LOOKBACK = 12
LIQ_LOOKBACK = 52
EVENT_WINDOW_DAYS = 14
RNG_SEED = 42

LLAMA = "https://api.llama.fi"
CHAINS = [
    "Ethereum",
    "Arbitrum",
    "Base",
    "Optimism",
    "Polygon",
    "Avalanche",
    "BSC",
    "Solana",
    "Tron",
]

# Trump / PolitiFi identifiers (public contracts + CoinGecko ids)
TRUMP_CG_ID = "official-trump"
WLFI_CG_ID = "world-liberty-financial"
TRUMP_MINT = "6p6xgHyF7AeE6TZkSmFsko444wqoP15icUSqi2jfGiPN"
WLFI_ETH = "0xda5e1988097297dcdc1f90d4dfe7909e847cbef6"
WLFI_LLAMA_SLUG = "world-liberty-financial"

DEFI_COLOR = "#14F195"
TVL_COLOR = "#6ea8ff"
VOL_COLOR = "#f0b90b"

print(f"DB:              {DB_PATH}")
print(f"FRED_API_KEY:    {'set' if FRED_API_KEY else 'NOT set'}")
print(f"CG_DEMO_API_KEY: {'set' if CG_DEMO_API_KEY else 'NOT set (CoinGecko needs Demo key)'}")
print(f"Defi cache:      {CACHE_DIR}")
print(f"Events CSV:      {EVENTS_CSV.exists()}")
print(f"Chains:          {', '.join(CHAINS)}")


DB:              /home/ricka/Git/GitHub/ccquant/data/ccquant.duckdb
FRED_API_KEY:    set
CG_DEMO_API_KEY: set
Defi cache:      /home/ricka/Git/GitHub/ccquant/data/defi_cache
Events CSV:      True
Chains:          Ethereum, Arbitrum, Base, Optimism, Polygon, Avalanche, BSC, Solana, Tron


## 1. Cache helpers + DefiLlama fetchers

Keyless DefiLlama endpoints (`api.llama.fi`). Global TVL excludes liquid staking /
double-counted TVL per Llama v2 historical chain TVL docs. DEX volume and fees use
overview `totalDataChart` series. 12h JSON cache under `data/defi_cache/`.


In [ ]:
def _cache_path(name: str) -> Path:
    return CACHE_DIR / f"{name}.json"


def _cache_fresh(path: Path) -> bool:
    if not path.exists():
        return False
    age_h = (time.time() - path.stat().st_mtime) / 3600.0
    return age_h < CACHE_MAX_AGE_H


def _read_cache(name: str) -> list[dict[str, Any]] | None:
    path = _cache_path(name)
    if not _cache_fresh(path):
        return None
    return json.loads(path.read_text())


def _write_cache(name: str, rows: list[dict[str, Any]]) -> None:
    _cache_path(name).write_text(json.dumps(rows))


def rows_to_df(rows: list[dict[str, Any]], value_col: str) -> pl.DataFrame:
    return pl.DataFrame(
        {
            "date": [date.fromisoformat(r["date"]) for r in rows],
            value_col: [float(r["value"]) for r in rows],
        }
    ).sort("date")


def _ts_to_iso(ts: int | float) -> str:
    return datetime.fromtimestamp(int(ts), tz=UTC).date().isoformat()


def fetch_llama_global_tvl() -> pl.DataFrame | None:
    cached = _read_cache("llama_global_tvl")
    if cached is not None:
        print("  global TVL: cache hit")
        return rows_to_df(cached, "tvl_usd")
    try:
        with httpx.Client(timeout=60.0) as client:
            resp = client.get(f"{LLAMA}/v2/historicalChainTvl")
            resp.raise_for_status()
            data = resp.json()
        rows = [
            {"date": _ts_to_iso(p["date"]), "value": float(p["tvl"])}
            for p in data
            if "date" in p and "tvl" in p
        ]
        if not rows:
            return None
        _write_cache("llama_global_tvl", rows)
        print(f"  global TVL: DefiLlama {len(rows)} pts")
        return rows_to_df(rows, "tvl_usd")
    except Exception as exc:
        print(f"  global TVL: FAILED ({exc})")
        return None


def fetch_llama_chain_tvl(chain: str) -> pl.DataFrame | None:
    key = f"llama_tvl_{chain.lower()}"
    cached = _read_cache(key)
    if cached is not None:
        print(f"  TVL {chain}: cache hit")
        return rows_to_df(cached, "tvl_usd").with_columns(pl.lit(chain).alias("chain"))
    try:
        with httpx.Client(timeout=60.0) as client:
            resp = client.get(f"{LLAMA}/v2/historicalChainTvl/{chain}")
            resp.raise_for_status()
            data = resp.json()
        rows = [
            {"date": _ts_to_iso(p["date"]), "value": float(p["tvl"])}
            for p in data
            if "date" in p and "tvl" in p
        ]
        if not rows:
            return None
        _write_cache(key, rows)
        print(f"  TVL {chain}: DefiLlama {len(rows)} pts")
        return rows_to_df(rows, "tvl_usd").with_columns(pl.lit(chain).alias("chain"))
    except Exception as exc:
        print(f"  TVL {chain}: FAILED ({exc})")
        return None


def _fetch_overview_chart(kind: str, data_type: str, value_col: str) -> pl.DataFrame | None:
    # kind in {dexs, fees}; data_type e.g. dailyVolume / dailyFees.
    key = f"llama_{kind}_{data_type}"
    cached = _read_cache(key)
    if cached is not None:
        print(f"  {kind}: cache hit")
        return rows_to_df(cached, value_col)
    url = (
        f"{LLAMA}/overview/{kind}"
        f"?excludeTotalDataChart=false&excludeTotalDataChartBreakdown=true"
        f"&dataType={data_type}"
    )
    try:
        with httpx.Client(timeout=90.0) as client:
            resp = client.get(url)
            resp.raise_for_status()
            payload = resp.json()
        chart = payload.get("totalDataChart") or []
        rows = [
            {"date": _ts_to_iso(pt[0]), "value": float(pt[1])}
            for pt in chart
            if isinstance(pt, (list, tuple)) and len(pt) >= 2
        ]
        if not rows:
            return None
        _write_cache(key, rows)
        print(f"  {kind}: DefiLlama {len(rows)} pts")
        return rows_to_df(rows, value_col)
    except Exception as exc:
        print(f"  {kind}: FAILED ({exc})")
        return None


def fetch_llama_dex_volume() -> pl.DataFrame | None:
    return _fetch_overview_chart("dexs", "dailyVolume", "dex_volume_usd")


def fetch_llama_fees() -> pl.DataFrame | None:
    return _fetch_overview_chart("fees", "dailyFees", "fees_usd")


def fetch_llama_protocol_tvl(slug: str) -> pl.DataFrame | None:
    key = f"llama_protocol_{slug}"
    cached = _read_cache(key)
    if cached is not None:
        print(f"  protocol {slug}: cache hit")
        return rows_to_df(cached, "protocol_tvl_usd")
    try:
        with httpx.Client(timeout=60.0) as client:
            resp = client.get(f"{LLAMA}/protocol/{slug}")
            resp.raise_for_status()
            payload = resp.json()
        series = payload.get("tvl") or []
        rows = [
            {"date": _ts_to_iso(p["date"]), "value": float(p["totalLiquidityUSD"])}
            for p in series
            if isinstance(p, dict) and "date" in p and "totalLiquidityUSD" in p
        ]
        if not rows:
            print(f"  protocol {slug}: empty TVL series (listed but no history)")
            return None
        _write_cache(key, rows)
        print(f"  protocol {slug}: DefiLlama {len(rows)} pts")
        return rows_to_df(rows, "protocol_tvl_usd")
    except Exception as exc:
        print(f"  protocol {slug}: FAILED ({exc})")
        return None


print("fetch helpers ready")


fetch helpers ready


## 2. Load aggregates (real or synthetic)

Pull Llama series. If any critical series fails, synthesize a coherent panel so
downstream regime logic still runs.


In [ ]:
def synthetic_defi_panel(n_days: int = 900) -> dict[str, pl.DataFrame]:
    # Seeded paths correlated with a shared cycle so the notebook completes offline.
    rng = np.random.default_rng(RNG_SEED)
    end = date.today()
    dates = [end - timedelta(days=n_days - i) for i in range(n_days)]
    t = np.arange(n_days)
    cycle = 0.15 * np.sin(2 * np.pi * t / 180.0) + 0.08 * np.sin(2 * np.pi * t / 420.0)
    shock = np.zeros(n_days)
    for i, d in enumerate(dates):
        if d == date(2022, 5, 9):
            shock[i : i + 20] -= 0.25
        if d == date(2022, 11, 11):
            shock[i : i + 25] -= 0.20
        if d == date(2025, 1, 20):
            shock[i : i + 15] += 0.12
    noise = 0.01 * rng.standard_normal(n_days).cumsum() / 10
    base_tvl = 80e9 * np.exp(0.00015 * t + cycle + shock + noise)
    dex = 4e9 * np.exp(0.0001 * t + 1.2 * cycle + shock + 2 * noise)
    fees_arr = 25e6 * np.exp(0.00012 * t + 1.1 * cycle + 0.8 * shock + 2 * noise)
    global_tvl_s = pl.DataFrame({"date": dates, "tvl_usd": base_tvl})
    dex_df = pl.DataFrame({"date": dates, "dex_volume_usd": dex})
    fees_df = pl.DataFrame({"date": dates, "fees_usd": fees_arr})
    shares = {
        "Ethereum": 0.45,
        "Arbitrum": 0.08,
        "Base": 0.07,
        "Optimism": 0.04,
        "Polygon": 0.05,
        "Avalanche": 0.04,
        "BSC": 0.08,
        "Solana": 0.12,
        "Tron": 0.07,
    }
    chain_frames = []
    for chain, share in shares.items():
        n2 = 1.0 + 0.05 * rng.standard_normal(n_days).cumsum() / 20
        chain_frames.append(
            pl.DataFrame(
                {
                    "date": dates,
                    "tvl_usd": base_tvl * share * np.clip(n2, 0.5, 1.5),
                    "chain": chain,
                }
            )
        )
    return {
        "global_tvl": global_tvl_s,
        "dex": dex_df,
        "fees": fees_df,
        "chain_tvl": pl.concat(chain_frames),
        "source": "synthetic",
    }


print("Fetching DefiLlama aggregates…")
global_tvl = fetch_llama_global_tvl()
dex_vol = fetch_llama_dex_volume()
fees = fetch_llama_fees()

chain_parts: list[pl.DataFrame] = []
for chain in CHAINS:
    cdf = fetch_llama_chain_tvl(chain)
    if cdf is not None:
        chain_parts.append(cdf)
    time.sleep(0.15)

if global_tvl is None or dex_vol is None or fees is None or not chain_parts:
    print("\nFalling back to SYNTHETIC DeFi panel (missing Llama series).")
    synth = synthetic_defi_panel()
    if global_tvl is None:
        global_tvl = synth["global_tvl"]
    if dex_vol is None:
        dex_vol = synth["dex"]
    if fees is None:
        fees = synth["fees"]
    if not chain_parts:
        chain_tvl = synth["chain_tvl"]
    else:
        chain_tvl = pl.concat(chain_parts)
    DEFI_SOURCE = "synthetic+partial" if chain_parts else "synthetic"
else:
    chain_tvl = pl.concat(chain_parts)
    DEFI_SOURCE = "defillama"

print(f"\nDeFi source tag: {DEFI_SOURCE}")
print(f"  global TVL: {global_tvl.height}  {global_tvl['date'].min()} -> {global_tvl['date'].max()}")
print(f"  DEX volume: {dex_vol.height}")
print(f"  fees:       {fees.height}")
print(f"  chain TVL:  {chain_tvl.height} rows / {chain_tvl['chain'].n_unique()} chains")


Fetching DefiLlama aggregates…
  global TVL: cache hit
  dexs: cache hit
  fees: cache hit
  TVL Ethereum: cache hit
  TVL Arbitrum: cache hit
  TVL Base: cache hit
  TVL Optimism: DefiLlama 1828 pts
  TVL Polygon: DefiLlama 2107 pts
  TVL Avalanche: DefiLlama 1990 pts
  TVL BSC: DefiLlama 2085 pts
  TVL Solana: cache hit
  TVL Tron: DefiLlama 2295 pts

DeFi source tag: defillama
  global TVL: 3214  2017-09-27 -> 2026-07-15
  DEX volume: 3725
  fees:       3034
  chain TVL:  18400 rows / 9 chains


## 3. Market context (DuckDB) + optional FRED liquidity

BTC/ETH weekly closes for beta context. Optional FRED M2 / Fed BS / real-rate
proxy when `FRED_API_KEY` is set.


In [ ]:
def to_weekly(
    df: pl.DataFrame, date_col: str = "date", value_cols: list[str] | None = None
) -> pl.DataFrame:
    value_cols = value_cols or [c for c in df.columns if c != date_col]
    return (
        df.sort(date_col)
        .with_columns(pl.col(date_col).dt.truncate(WEEKLY_FREQ).alias("week"))
        .group_by("week")
        .agg([pl.col(c).last().alias(c) for c in value_cols])
        .rename({"week": "date"})
        .sort("date")
    )


btc_w = eth_w = None
try:
    panel = load_daily_panel(DB_PATH)
    print(f"OHLCV panel: {panel.shape}")
    for sym, holder in (("BTC", "btc"), ("ETH", "eth")):
        sub = panel.filter(pl.col("symbol") == sym).sort("date").unique(subset=["date"])
        if sub.height == 0:
            print(f"  {sym}: MISSING in DuckDB")
            continue
        w = to_weekly(sub.select(["date", "close", "volume"]), value_cols=["close", "volume"])
        w = w.rename({"close": f"{sym.lower()}_close", "volume": f"{sym.lower()}_volume"})
        if holder == "btc":
            btc_w = w
        else:
            eth_w = w
        print(f"  {sym} weekly: {w.height}  {w['date'].min()} -> {w['date'].max()}")
except Exception as exc:
    print(f"DuckDB panel unavailable ({exc})")

oi_w = None
try:
    sig = load_signals_panel(DB_PATH)
    if "total_open_interest_usd" in sig.columns:
        oi = (
            sig.select(["date", "total_open_interest_usd"])
            .drop_nulls()
            .group_by("date")
            .agg(pl.col("total_open_interest_usd").mean())
            .sort("date")
        )
        oi_w = to_weekly(oi, value_cols=["total_open_interest_usd"])
        print(f"  OI weekly: {oi_w.height}")
except Exception as exc:
    print(f"  OI: skipped ({exc})")


def fetch_fred(series_id: str) -> pl.DataFrame | None:
    if not FRED_API_KEY:
        return None
    try:
        with httpx.Client(timeout=45.0) as client:
            resp = client.get(
                "https://api.stlouisfed.org/fred/series/observations",
                params={
                    "series_id": series_id,
                    "api_key": FRED_API_KEY,
                    "file_type": "json",
                    "observation_start": "2018-01-01",
                },
            )
            resp.raise_for_status()
            obs = resp.json().get("observations", [])
        rows = [
            {"date": date.fromisoformat(o["date"]), "value": float(o["value"])}
            for o in obs
            if o.get("value") not in (".", None, "")
        ]
        return pl.DataFrame(rows).sort("date") if rows else None
    except Exception as exc:
        print(f"  FRED {series_id}: FAILED ({exc})")
        return None


fred_parts: list[pl.DataFrame] = []
for sid, col in (("M2SL", "m2"), ("WALCL", "fed_bs"), ("DGS10", "dgs10"), ("T10YIE", "t10yie")):
    df = fetch_fred(sid)
    if df is not None:
        fred_parts.append(df.rename({"value": col}))
        print(f"  FRED {sid}: {df.height} pts")
    else:
        print(f"  FRED {sid}: MISSING")

liq_w = None
if fred_parts:
    liq = fred_parts[0]
    for part in fred_parts[1:]:
        liq = liq.join_asof(part.sort("date"), on="date", strategy="backward")
    if "dgs10" in liq.columns and "t10yie" in liq.columns:
        liq = liq.with_columns((pl.col("dgs10") - pl.col("t10yie")).alias("real_10y"))
    liq_w = to_weekly(liq, value_cols=[c for c in liq.columns if c != "date"])
    print(f"  liquidity weekly: {liq_w.height}")
else:
    print("  liquidity: skipped (no FRED key)")


OHLCV panel: (90, 8)
  BTC weekly: 5  2026-06-08 -> 2026-07-06
  ETH weekly: 5  2026-06-08 -> 2026-07-06
  OI weekly: 5
  FRED M2SL: 101 pts
  FRED WALCL: 445 pts
  FRED DGS10: 2131 pts
  FRED T10YIE: 2132 pts
  liquidity weekly: 101


## 4. Weekly DeFi features, concentration, activity regime

Build a weekly spine of global TVL / DEX volume / fees, chain top-3 share and HHI,
momentum, and an **activity composite**. Regime = sign of 12-week composite momentum.


In [ ]:
def z_expr(col: str, win: int = LIQ_LOOKBACK) -> pl.Expr:
    x = pl.col(col)
    mu = x.rolling_mean(win, min_samples=max(8, win // 4))
    sd = x.rolling_std(win, min_samples=max(8, win // 4))
    return ((x - mu) / sd).alias(f"z_{col}")


tvl_w = to_weekly(global_tvl, value_cols=["tvl_usd"])
dex_w = to_weekly(dex_vol, value_cols=["dex_volume_usd"])
fees_w = to_weekly(fees, value_cols=["fees_usd"])

chain_w = (
    chain_tvl.sort("date")
    .with_columns(pl.col("date").dt.truncate(WEEKLY_FREQ).alias("week"))
    .group_by(["week", "chain"])
    .agg(pl.col("tvl_usd").last())
    .rename({"week": "date"})
    .sort("date")
)


def concentration(df: pl.DataFrame) -> pl.DataFrame:
    rows = []
    for key, g in df.group_by("date", maintain_order=True):
        day = key[0] if isinstance(key, tuple) else key
        vals = g["tvl_usd"].to_numpy()
        total = float(vals.sum()) if len(vals) else 0.0
        if total <= 0:
            continue
        shares = np.sort(vals / total)[::-1]
        rows.append(
            {
                "date": day,
                "top3_share": float(shares[:3].sum()),
                "hhi": float(np.sum(shares**2)),
                "n_chains": len(shares),
            }
        )
    if not rows:
        return pl.DataFrame(
            schema={
                "date": pl.Date,
                "top3_share": pl.Float64,
                "hhi": pl.Float64,
                "n_chains": pl.Int64,
            }
        )
    return pl.DataFrame(rows).sort("date")


conc = concentration(chain_w)

feat = (
    tvl_w.join(dex_w, on="date", how="inner")
    .join(fees_w, on="date", how="inner")
    .join(conc, on="date", how="left")
    .sort("date")
)

if btc_w is not None:
    feat = feat.join_asof(btc_w.sort("date"), on="date", strategy="backward")
if eth_w is not None:
    feat = feat.join_asof(eth_w.sort("date"), on="date", strategy="backward")
if oi_w is not None:
    feat = feat.join_asof(oi_w.sort("date"), on="date", strategy="backward")
if liq_w is not None:
    feat = feat.join_asof(liq_w.sort("date"), on="date", strategy="backward")

feat = (
    feat.with_columns(
        [
            pl.col("tvl_usd").log().diff(MOM_LOOKBACK).alias("tvl_mom"),
            pl.col("dex_volume_usd").log().diff(MOM_LOOKBACK).alias("dex_vol_mom"),
            pl.col("fees_usd").log().diff(MOM_LOOKBACK).alias("fees_mom"),
            pl.col("tvl_usd").pct_change(4).alias("tvl_chg_4w"),
            pl.col("tvl_usd").pct_change(13).alias("tvl_chg_13w"),
            pl.col("dex_volume_usd").pct_change(4).alias("dex_chg_4w"),
            pl.col("dex_volume_usd").pct_change(13).alias("dex_chg_13w"),
        ]
    )
    .with_columns([z_expr("tvl_mom"), z_expr("dex_vol_mom"), z_expr("fees_mom")])
    .with_columns(
        (
            pl.col("z_tvl_mom").fill_null(0)
            + pl.col("z_dex_vol_mom").fill_null(0)
            + pl.col("z_fees_mom").fill_null(0)
        ).alias("activity_composite")
    )
    .with_columns(
        [
            pl.col("activity_composite").diff(MOM_LOOKBACK).alias("activity_mom"),
            (pl.col("activity_composite").diff(MOM_LOOKBACK) > 0)
            .cast(pl.Int8)
            .alias("regime"),
        ]
    )
)

latest = feat["date"].max()
chain_latest = (
    chain_w.sort("date")
    .with_columns(pl.col("tvl_usd").pct_change(13).over("chain").alias("tvl_chg_13w"))
    .filter(pl.col("date") == latest)
    .sort("tvl_chg_13w", descending=True)
)

print(f"Feature frame: {feat.shape}  {feat['date'].min()} -> {feat['date'].max()}")
print(f"Latest week: {latest}")
print("Chain 13w TVL leaders/laggards:")
print(chain_latest.select(["chain", "tvl_usd", "tvl_chg_13w"]))
feat.tail(3)


Feature frame: (434, 30)  2018-03-26 -> 2026-07-13
Latest week: 2026-07-13
Chain 13w TVL leaders/laggards:
shape: (9, 3)
┌───────────┬──────────────┬─────────────┐
│ chain     ┆ tvl_usd      ┆ tvl_chg_13w │
│ ---       ┆ ---          ┆ ---         │
│ str       ┆ f64          ┆ f64         │
╞═══════════╪══════════════╪═════════════╡
│ Base      ┆ 4.5783e9     ┆ 0.021371    │
│ Tron      ┆ 4.8113e9     ┆ -0.060556   │
│ BSC       ┆ 4.9705e9     ┆ -0.099907   │
│ Solana    ┆ 4.9140e9     ┆ -0.154005   │
│ Ethereum  ┆ 4.1228e10    ┆ -0.203169   │
│ Optimism  ┆ 3.03741442e8 ┆ -0.208236   │
│ Polygon   ┆ 9.11969655e8 ┆ -0.275598   │
│ Avalanche ┆ 4.72795255e8 ┆ -0.337646   │
│ Arbitrum  ┆ 1.2719e9     ┆ -0.344331   │
└───────────┴──────────────┴─────────────┘


date,tvl_usd,dex_volume_usd,fees_usd,top3_share,hhi,n_chains,btc_close,btc_volume,eth_close,eth_volume,total_open_interest_usd,m2,fed_bs,dgs10,t10yie,real_10y,tvl_mom,dex_vol_mom,fees_mom,tvl_chg_4w,tvl_chg_13w,dex_chg_4w,dex_chg_13w,z_tvl_mom,z_dex_vol_mom,z_fees_mom,activity_composite,activity_mom,regime
date,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8
2026-06-29,7.4357e10,5.1857e9,5.0311277e7,0.807466,0.439892,9,50000.0,1000.0,3000.0,1000.0,1e6,23052.3,6.69995e6,4.39,2.48,1.91,-0.260469,0.000684,0.073023,0.071645,-0.189351,-0.178572,0.179174,-0.774812,0.250375,0.405037,-0.119399,2.68194,1
2026-07-06,7.3575e10,5.7083e9,5.4145996e7,0.80262,0.439227,9,50000.0,1000.0,3000.0,1000.0,1e6,23052.3,6.69995e6,4.39,2.48,1.91,-0.230307,-0.471143,-0.13981,0.01912,-0.237419,0.13566,0.101536,-0.624281,-0.769493,-0.43428,-1.828054,-0.575897,0
2026-07-13,7.5587e10,6.9198e9,3.3412317e7,0.805397,0.445806,9,50000.0,1000.0,3000.0,1000.0,1e6,23052.3,6.69995e6,4.39,2.48,1.91,-0.102744,0.328338,-0.407011,0.031998,-0.183994,0.436326,-0.243214,-0.06722,1.117167,-1.497042,-0.447095,3.994004,1


## 5. Visualize aggregates + regimes

Global TVL vs DEX volume with expanding-regime shading; chain TVL stacked share;
activity composite vs BTC when available.


In [ ]:
reg = feat.select(["date", "regime"]).drop_nulls()
spans: list[tuple[date, date, int]] = []
if reg.height:
    start = reg["date"][0]
    cur = int(reg["regime"][0])
    for i in range(1, reg.height):
        r = int(reg["regime"][i])
        if r != cur:
            spans.append((start, reg["date"][i], cur))
            start = reg["date"][i]
            cur = r
    spans.append((start, reg["date"][-1], cur))

fig = make_subplots(specs=[[{"secondary_y": True}]])
for s, e, r in spans:
    if r == 1:
        # ISO strings avoid Plotly date/int mixups in axis-spanning shapes
        fig.add_vrect(
            x0=s.isoformat() if isinstance(s, date) else s,
            x1=e.isoformat() if isinstance(e, date) else e,
            fillcolor="rgba(20,241,149,0.08)",
            line_width=0,
            layer="below",
        )

fig.add_trace(
    go.Scatter(
        x=feat["date"], y=feat["tvl_usd"], name="Global DeFi TVL",
        line=dict(color=TVL_COLOR, width=2),
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=feat["date"], y=feat["dex_volume_usd"], name="DEX volume (weekly last)",
        line=dict(color=VOL_COLOR, width=1.5),
    ),
    secondary_y=True,
)
fig.update_layout(
    template="plotly_dark",
    title="Global DeFi TVL vs DEX volume (green = expanding activity regime)",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    height=480,
)
fig.update_yaxes(title_text="TVL USD", secondary_y=False, type="log")
fig.update_yaxes(title_text="DEX volume USD", secondary_y=True, type="log")
fig.show()

pivot = chain_w.pivot(on="chain", index="date", values="tvl_usd").sort("date")
fig2 = go.Figure()
for chain in CHAINS:
    if chain not in pivot.columns:
        continue
    fig2.add_trace(
        go.Scatter(
            x=pivot["date"], y=pivot[chain], name=chain,
            stackgroup="one", groupnorm="percent",
        )
    )
fig2.update_layout(
    template="plotly_dark",
    title="Chain TVL share (% of tracked set)",
    height=420,
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig2.show()

fig3 = make_subplots(specs=[[{"secondary_y": True}]])
fig3.add_trace(
    go.Scatter(
        x=feat["date"], y=feat["activity_composite"], name="Activity composite",
        line=dict(color=DEFI_COLOR, width=2),
    ),
    secondary_y=False,
)
if "btc_close" in feat.columns:
    fig3.add_trace(
        go.Scatter(
            x=feat["date"], y=feat["btc_close"], name="BTC",
            line=dict(color="#f7931a", width=1.5),
        ),
        secondary_y=True,
    )
fig3.update_layout(
    template="plotly_dark",
    title="DeFi activity composite vs BTC",
    height=420,
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig3.update_yaxes(title_text="Composite (z-sum)", secondary_y=False)
fig3.update_yaxes(title_text="BTC USD", type="log", secondary_y=True)
fig3.show()


## 6. Legislation & regime shocks

Load curated events from `dbt/seeds/events.csv`. Mark on the TVL chart and compute
±14-day pre/post changes in TVL and DEX volume.


In [ ]:
events = pl.read_csv(EVENTS_CSV, try_parse_dates=True)
events_use = events.filter(
    pl.col("asset_scope").is_in(["DEFI", "BTC"])
    | pl.col("category").is_in(
        ["regulatory", "legislative", "market", "political", "monetary"]
    )
).sort("date")
print(f"Events loaded: {events_use.height}")
print(
    events_use.select(
        ["date", "category", "title", "asset_scope", "anticipated_effect_direction"]
    )
)

# Plotly axis-spanning annotations break on datetime.date (sum starts at 0).
# Use ISO strings for shapes + separate annotations (matches BTC.ipynb pattern).
fig_e = go.Figure()
fig_e.add_trace(
    go.Scatter(
        x=[d.isoformat() if hasattr(d, "isoformat") else d for d in feat["date"].to_list()],
        y=feat["tvl_usd"].to_list(),
        name="Global TVL",
        line=dict(color=TVL_COLOR, width=2),
    )
)
for row in events_use.iter_rows(named=True):
    d = row["date"]
    if isinstance(d, datetime):
        d = d.date()
    x = d.isoformat() if isinstance(d, date) else str(d)
    color = {
        "positive": "#14F195",
        "negative": "#e84142",
        "neutral": "#aaaaaa",
    }.get(str(row["anticipated_effect_direction"]), "#aaaaaa")
    fig_e.add_vline(x=x, line_dash="dash", line_color=color, opacity=0.7)
    fig_e.add_annotation(
        x=x,
        y=1.02,
        yref="paper",
        text=str(row["title"])[:28],
        showarrow=False,
        font=dict(size=9, color=color),
        xanchor="left",
    )
fig_e.update_layout(
    template="plotly_dark",
    title="Global DeFi TVL with curated shock markers",
    yaxis_type="log",
    height=480,
)
fig_e.show()

g_daily = global_tvl.sort("date")
d_daily = dex_vol.sort("date")


def window_change(
    series: pl.DataFrame, col: str, event_date: date, days: int = EVENT_WINDOW_DAYS
) -> float | None:
    pre = series.filter(
        (pl.col("date") >= event_date - timedelta(days=days))
        & (pl.col("date") < event_date)
    )
    post = series.filter(
        (pl.col("date") > event_date)
        & (pl.col("date") <= event_date + timedelta(days=days))
    )
    if pre.height == 0 or post.height == 0:
        return None
    a = float(pre[col].mean())
    b = float(post[col].mean())
    if a == 0:
        return None
    return b / a - 1.0


impact_rows = []
for row in events_use.iter_rows(named=True):
    d = row["date"]
    if isinstance(d, datetime):
        d = d.date()
    impact_rows.append(
        {
            "date": d,
            "title": row["title"],
            "direction": row["anticipated_effect_direction"],
            "tvl_pre_post": window_change(g_daily, "tvl_usd", d),
            "dex_pre_post": window_change(d_daily, "dex_volume_usd", d),
        }
    )

impact = pl.DataFrame(impact_rows)
print("+/-14d mean level change (post/pre - 1):")
print(impact)


## 7. Outlook snapshot (rule-based)

Not a price model — a structured read of current regime, momentum, chain rotation,
and whether recent events align with the activity impulse.


In [ ]:
last = feat.drop_nulls(subset=["activity_composite"]).tail(1)
assert last.height == 1
row = last.to_dicts()[0]
regime = int(row.get("regime") or 0)
mom4 = row.get("tvl_chg_4w")
mom13 = row.get("tvl_chg_13w")
dex4 = row.get("dex_chg_4w")
dex13 = row.get("dex_chg_13w")
comp = row.get("activity_composite")
top3 = row.get("top3_share")
hhi = row.get("hhi")

leaders = chain_latest.head(3)["chain"].to_list() if chain_latest.height else []
laggards = chain_latest.tail(3)["chain"].to_list() if chain_latest.height else []

cutoff = date.today() - timedelta(days=90)
recent_ev = events_use.filter(pl.col("date") >= cutoff)


def sgn(x: float | None) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "n/a"
    if x > 0.02:
        return "up"
    if x < -0.02:
        return "down"
    return "flat"


print("=" * 64)
print("DEFI OUTLOOK SNAPSHOT")
print("=" * 64)
print(f"As-of week:           {row['date']}")
print(f"Source tag:           {DEFI_SOURCE}")
print(f"Activity regime:      {'EXPANDING' if regime == 1 else 'CONTRACTING'}")
if comp is not None:
    print(f"Activity composite:   {comp:.2f}")
print(f"TVL 4w / 13w:         {sgn(mom4)} / {sgn(mom13)}  ({mom4}, {mom13})")
print(f"DEX 4w / 13w:         {sgn(dex4)} / {sgn(dex13)}  ({dex4}, {dex13})")
if top3 is not None:
    print(f"Concentration top3:   {top3:.1%}")
if hhi is not None:
    print(f"HHI:                  {hhi:.3f}")
print(f"13w TVL leaders:      {', '.join(leaders)}")
print(f"13w TVL laggards:     {', '.join(laggards)}")
print(f"Events (last 90d):    {recent_ev.height}")
for er in recent_ev.iter_rows(named=True):
    print(f"  - {er['date']}: {er['title']} [{er['anticipated_effect_direction']}]")

bullish = regime == 1 and sgn(mom13) == "up" and sgn(dex13) in {"up", "flat"}
bearish = regime == 0 and sgn(mom13) == "down"
if bullish:
    outlook = (
        "Activity impulse is expanding with constructive 13-week TVL/DEX momentum. "
        "Bias: constructive for DeFi risk until composite momentum flips or a negative "
        "regulatory shock prints."
    )
elif bearish:
    outlook = (
        "Activity regime is contracting with soft 13-week TVL momentum. "
        "Bias: defensive — prefer waiting for composite momentum to turn positive "
        "or for post-shock stabilization in DEX volume."
    )
else:
    outlook = (
        "Mixed signal: regime and multi-horizon momentum disagree. "
        "Bias: range / rotation — watch chain leaders vs laggards and event windows "
        "rather than a single directional call."
    )
print("\nOutlook:")
print(outlook)
print("=" * 64)


## 8. Can we track Trump's assets? (crypto-native)

| Layer | Feasible? | How |
|---|---|---|
| **$TRUMP** (Solana memecoin) | Yes | DuckDB OHLCV (Coinbase/CG) + CoinGecko `official-trump` |
| **WLFI** | Yes (price) | DuckDB OHLCV + CoinGecko `world-liberty-financial` |
| **Related protocol TVL** | Partial | DefiLlama `world-liberty-financial` is listed but empty; fall back to CG mcap + USD1 (`usd1-wlfi`) + Lorenzo `lorenzo-susd1+` |
| **Public token mints** | Yes | Wallet seeds: TRUMP + WLFI Solana mints |
| **Personal / treasury wallets** | Only if publicly attributed | Do not invent addresses |
| **DJT equity / TradFi** | No | Out of scope for ccquant |

Populate DuckDB with:
`include_symbols: [TRUMP, WLFI]` then `uv run ccquant sync backfill --interval 1d --force`
(or the one-shot seed already run for this DB).

Data provided by [CoinGecko](https://www.coingecko.com) when CG series are used.


In [ ]:
def fetch_coingecko_market_chart(coin_id: str, days: int = 365) -> pl.DataFrame | None:
    key = f"cg_{coin_id}_{days}"
    cached = _read_cache(key)
    if cached is not None:
        print(f"  CG {coin_id}: cache hit")
        return pl.DataFrame(
            {
                "date": [date.fromisoformat(r["date"]) for r in cached],
                "price": [float(r["price"]) for r in cached],
                "mcap": [float(r["mcap"]) for r in cached],
                "volume": [float(r["volume"]) for r in cached],
            }
        ).sort("date")
    if not CG_DEMO_API_KEY:
        print(f"  CG {coin_id}: skipped — set CG_DEMO_API_KEY")
        return None
    headers = {"x-cg-demo-api-key": CG_DEMO_API_KEY}
    try:
        with httpx.Client(timeout=60.0, headers=headers) as client:
            resp = client.get(
                f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart",
                params={"vs_currency": "usd", "days": days, "interval": "daily"},
            )
            resp.raise_for_status()
            payload = resp.json()
        prices = {int(p[0]): float(p[1]) for p in payload.get("prices", [])}
        mcaps = {int(p[0]): float(p[1]) for p in payload.get("market_caps", [])}
        vols = {int(p[0]): float(p[1]) for p in payload.get("total_volumes", [])}
        rows = []
        for ts, px in sorted(prices.items()):
            rows.append(
                {
                    "date": datetime.fromtimestamp(ts / 1000.0, tz=UTC).date().isoformat(),
                    "price": px,
                    "mcap": mcaps.get(ts, float("nan")),
                    "volume": vols.get(ts, float("nan")),
                }
            )
        if not rows:
            return None
        _write_cache(key, rows)
        print(f"  CG {coin_id}: {len(rows)} pts  ({COINGECKO_ATTRIBUTION})")
        return pl.DataFrame(
            {
                "date": [date.fromisoformat(r["date"]) for r in rows],
                "price": [float(r["price"]) for r in rows],
                "mcap": [float(r["mcap"]) for r in rows],
                "volume": [float(r["volume"]) for r in rows],
            }
        ).sort("date")
    except Exception as exc:
        print(f"  CG {coin_id}: FAILED ({exc})")
        return None


print("Trump crypto panel…")
trump_px = fetch_coingecko_market_chart(TRUMP_CG_ID, days=365)
wlfi_px = fetch_coingecko_market_chart(WLFI_CG_ID, days=365)
usd1_px = fetch_coingecko_market_chart("usd1-wlfi", days=365)
wlfi_tvl = fetch_llama_protocol_tvl(WLFI_LLAMA_SLUG)
# Related DeFi TVL: Lorenzo sUSD1+ (USD1 yield) when WLFI protocol TVL is empty
lorenzo_tvl = fetch_llama_protocol_tvl("lorenzo-susd1+")

trump_db = wlfi_db = None
try:
    # Prefer raw ohlcv_daily for full history — dbt fct_ohlcv_daily may be
    # incremental/short even after backfill.
    import duckdb

    with duckdb.connect(DB_PATH, read_only=True) as con:
        raw = pl.from_arrow(
            con.execute(
                """
                select symbol, date, close, volume, source
                from ohlcv_daily
                where symbol in ('TRUMP', 'WLFI')
                order by symbol, date, source
                """
            ).to_arrow_table()
        )
    for sym in ("TRUMP", "WLFI"):
        sub = (
            raw.filter(pl.col("symbol") == sym)
            .unique(subset=["date"], keep="last")
            .sort("date")
        )
        if sub.height:
            src = sub["source"][-1]
            print(
                f"  DuckDB {sym}: {sub.height} rows  "
                f"{sub['date'].min()} -> {sub['date'].max()}  source={src}"
            )
            frame = sub.select(["date", "close", "volume"])
            if sym == "TRUMP":
                trump_db = frame
            else:
                wlfi_db = frame
        else:
            print(
                f"  DuckDB {sym}: MISSING — seed asset + "
                "`uv run ccquant sync backfill --interval 1d --force`"
            )
except Exception as exc:
    print(f"  DuckDB Trump symbols: skipped ({exc})")

WLFI_SOL_MINT = "WLFinEv6ypjkczcS83FZqFpgFZYwQXutRbxGe7oC16g"
trump_wallet_rows = pl.DataFrame()
if WALLET_SEED.exists():
    wr = pl.read_csv(WALLET_SEED)
    trump_wallet_rows = wr.filter(
        pl.col("address").is_in([TRUMP_MINT, WLFI_SOL_MINT])
        | (pl.col("address").str.to_lowercase() == WLFI_ETH.lower())
        | pl.col("label")
        .str.to_lowercase()
        .str.contains("trump|wlfi|world.?liberty")
    )
    print(f"\nWallet seed matches: {trump_wallet_rows.height}")
    if trump_wallet_rows.height:
        print(trump_wallet_rows.select(["address", "chain", "label", "entity_type"]))
else:
    print("Wallet seed file MISSING")

# Prefer DuckDB OHLCV when present; CG remains secondary / mcap source
trump_price = trump_db.rename({"close": "price"}) if trump_db is not None else trump_px
wlfi_price = wlfi_db.rename({"close": "price"}) if wlfi_db is not None else wlfi_px

wlfi_tvl_source = "defillama"
if wlfi_tvl is None and wlfi_px is not None and "mcap" in wlfi_px.columns:
    wlfi_tvl = wlfi_px.select(
        ["date", pl.col("mcap").alias("protocol_tvl_usd")]
    ).drop_nulls()
    wlfi_tvl_source = "coingecko_mcap_proxy"
    print(
        f"  WLFI protocol TVL: using CoinGecko market-cap proxy "
        f"({wlfi_tvl.height} pts) — DefiLlama history empty"
    )
elif wlfi_tvl is not None:
    print(f"  WLFI protocol TVL: DefiLlama {wlfi_tvl.height} pts")

status = {
    "TRUMP_price": "REAL" if trump_price is not None else "MISSING",
    "TRUMP_duckdb": "REAL" if trump_db is not None else "MISSING",
    "WLFI_price": "REAL" if wlfi_price is not None else "MISSING",
    "WLFI_duckdb": "REAL" if wlfi_db is not None else "MISSING",
    "WLFI_protocol_TVL": (
        f"REAL({wlfi_tvl_source})" if wlfi_tvl is not None else "MISSING/EMPTY"
    ),
    "USD1_stablecoin": "REAL" if usd1_px is not None else "MISSING",
    "Lorenzo_sUSD1+_TVL": "REAL" if lorenzo_tvl is not None else "MISSING",
    "token_mint_seeds": (
        "REAL" if trump_wallet_rows.height > 0 else "MISSING"
    ),
    "DJT_equity": "OUT_OF_SCOPE",
}
print("\nAvailability:")
for k, v in status.items():
    print(f"  {k}: {v}")

fig_t = make_subplots(
    rows=2, cols=1, shared_xaxes=True, subplot_titles=("$TRUMP", "WLFI")
)
if trump_price is not None:
    src = "DB" if trump_db is not None else "CG"
    fig_t.add_trace(
        go.Scatter(
            x=[d.isoformat() if hasattr(d, "isoformat") else d for d in trump_price["date"].to_list()],
            y=trump_price["price"].to_list(),
            name=f"TRUMP ({src})",
            line=dict(color="#c41e3a"),
        ),
        row=1, col=1,
    )
if wlfi_price is not None:
    src = "DB" if wlfi_db is not None else "CG"
    fig_t.add_trace(
        go.Scatter(
            x=[d.isoformat() if hasattr(d, "isoformat") else d for d in wlfi_price["date"].to_list()],
            y=wlfi_price["price"].to_list(),
            name=f"WLFI ({src})",
            line=dict(color="#1e90ff"),
        ),
        row=2, col=1,
    )
fig_t.update_layout(
    template="plotly_dark",
    title="Trump-linked crypto tokens (prices)",
    height=520,
    legend=dict(orientation="h"),
)
fig_t.update_yaxes(type="log", row=1, col=1)
fig_t.update_yaxes(type="log", row=2, col=1)
fig_t.show()

fig_w = go.Figure()
if wlfi_tvl is not None:
    fig_w.add_trace(
        go.Scatter(
            x=[d.isoformat() if hasattr(d, "isoformat") else d for d in wlfi_tvl["date"].to_list()],
            y=wlfi_tvl["protocol_tvl_usd"].to_list(),
            name=f"WLFI ({wlfi_tvl_source})",
            line=dict(color="#1e90ff"),
        )
    )
if lorenzo_tvl is not None:
    fig_w.add_trace(
        go.Scatter(
            x=[d.isoformat() if hasattr(d, "isoformat") else d for d in lorenzo_tvl["date"].to_list()],
            y=lorenzo_tvl["protocol_tvl_usd"].to_list(),
            name="Lorenzo sUSD1+ TVL",
            line=dict(color="#14F195"),
        )
    )
if usd1_px is not None:
    fig_w.add_trace(
        go.Scatter(
            x=[d.isoformat() if hasattr(d, "isoformat") else d for d in usd1_px["date"].to_list()],
            y=usd1_px["mcap"].to_list(),
            name="USD1 mcap (CG)",
            line=dict(color="#f0b90b", dash="dot"),
        )
    )
if fig_w.data:
    fig_w.update_layout(
        template="plotly_dark",
        title="WLFI / USD1 activity proxies (DefiLlama TVL empty for WLFI protocol)",
        height=400,
        yaxis_type="log",
        legend=dict(orientation="h"),
    )
    fig_w.show()
else:
    print("No WLFI/USD1 TVL or mcap series available.")

print(
    "\nVerdict: crypto-native Trump exposure is trackable via $TRUMP + WLFI "
    "(DuckDB OHLCV and/or CoinGecko), USD1 stablecoin mcap, and public mint seeds. "
    "DefiLlama lists World Liberty Financial but currently publishes no TVL history — "
    "use mcap / USD1 / Lorenzo sUSD1+ as proxies. Personal wallets and DJT stock are out of scope."
)


## Caveats

- DefiLlama v2 historical chain TVL **excludes liquid staking and double-counted TVL**;
  levels are not identical to the DefiLlama UI "total" in every view.
- DEX volume / fees overview charts are **all-protocol aggregates**; chain rotation
  in §4 uses TVL shares of the tracked set only.
- Cache TTL is **12 hours** under `data/defi_cache/` (gitignored).
- Synthetic fallbacks are for pipeline continuity — do not publish as research fact.
- Event impacts are **associative** (±14d means), not causal identification.
- Trump panel: only **public** contracts/mints; no claim that seed rows are personal
  holdings. **DJT / TradFi out of scope.**
- Wallet sync chains today are Solana / Arbitrum / Bitcoin — WLFI's Ethereum L1
  contract is not tailed by `sync wallets` unless bridged coverage is added later.
